Before you submit this problem, make sure everything runs as expected. First, **restart the kernel** (in the menubar, select Kernel$\rightarrow$Restart) and then **run all cells** (in the menubar, select Cell$\rightarrow$Run All).

Make sure you fill in any place that says `YOUR CODE HERE` or "YOUR ANSWER HERE", as well as your name below:

In [ ]:
NAME = ""

In [ ]:
import IPython
assert IPython.version_info[0] >= 3, "Your version of IPython is too old, please update it."

---

In [ ]:
import hashlib

import cv2
import numpy as np
import seaborn as sns
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

from skimage import img_as_float32
from skimage.io import imread
from skimage.color import rgb2gray
from skimage.transform import rotate, rescale, warp, AffineTransform
from skimage.util import view_as_windows

from scipy.signal import wiener
from scipy.fftpack import dct, idct

In [ ]:
def read_as_RGB_and_GRAYSCALE(image_path):
    image = cv2.imread(image_path, cv2.IMREAD_UNCHANGED)
    image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    image_gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    return image_rgb, image_gray

In [ ]:
lenna, lenna_gray = read_as_RGB_and_GRAYSCALE("lenna.png")
baboon, baboon_gray = read_as_RGB_and_GRAYSCALE("baboon.png")

plt.imshow(np.hstack([lenna, baboon]), cmap="gray")

In [ ]:
def minmax_norm(image):
    """
    Normalizes the input image to the range [0, 1]

    Parameters:
    - image: Input image to be normalized

    Returns:
        Normalized image
    """
    return (image - image.min()) / (image.max() - image.min())

---
## Part A, PSNR and different noises (4 points)

$PSNR = 10 \cdot \log_{10} \left( \frac{MAX\_I^2}{\sigma^2} \right)$


Calculate MSE (Mean Squared Error) between two images.

In [ ]:
def calculate_psnr(image_original, image_noisy):
    """

    Parameters:
    - image_original: Original image.
    - image_noisy: Noisy image.

    Returns:
        Peak Signal-to-Noise Ratio (PSNR) between the original and noisy images.

    Read more about PSNR here: https://en.wikipedia.org/wiki/Peak_signal-to-noise_ratio
    """

    # YOUR CODE HERE
    raise NotImplementedError()
    return psnr

For each pixel, generate a random number. If it's < prob/2, set pixel to 0 (pepper). If between prob/2 and prob, set to max intensity (salt). Leave pixel unchanged otherwise.

Hint: use masking and np.random.rand() with thresholding.

In [ ]:
def add_salt_pepper_noise(image, salt_prob, pepper_prob):
    """
    Add salt and pepper noise to an image.

    Parameters:
    - image
    - salt_prob: Probability of adding salt noise.
    - pepper_prob: Probability of adding pepper noise.

    Returns:
        Noisy image

    Read more about salt and pepper noise here: https://en.wikipedia.org/wiki/Salt-and-pepper_noise
    """

    image_noisy = np.copy(image)

    # YOUR CODE HERE
    raise NotImplementedError()

    return image_noisy

Generate white Gaussian noise with mean 0 and specified standard deviation (std). Add this noise to the original image pixels.

Hint: clip the resulting values to the valid image range (e.g., 0-255 for 8-bit images) to avoid overflow/underflow (np.clip).

In [ ]:
def add_awgn_noise(image, noise_std):
    """
    Adds AWGN noise to image based on the specified noise std.

    Parameters:
    - image
    - noise_std: Standard deviation of the AWGN noise

    Returns:
        Image with added AWGN noise

    Read more about AWGN noise here: https://en.wikipedia.org/wiki/Additive_white_Gaussian_noise
    """

    # YOUR CODE HERE
    raise NotImplementedError()

    return image_noisy

Calculate target MSE using target PSNR using formula: $MSE = \frac{MAX\_I^2}{10^{PSNR/10}}$ 

Relation between variance and PSNR: $PSNR = 10 \cdot \log_{10} \left( \frac{MAX\_I^2}{\sigma^2} \right)$

Generate white Gaussian noise with mean 0 and std from MSE. Add noise to image, ensuring result stays in valid range. MAX_I is the max pixel value.

In [ ]:
def add_awgn_noise_with_target_psnr(image, target_psnr):
    """
    Adds AWGN noise to image to achieve a target PSNR.

    Parameters:
    - image
    - target_psnr: Target PSNR in dB

    Returns:
        Image with added AWGN noise to achieve the target PSNR

    Read more about PSNR here: https://en.wikipedia.org/wiki/Peak_signal-to-noise_ratio
    """
    # YOUR CODE HERE
    raise NotImplementedError()

    return image_noisy

In [ ]:
def denoise_with_median_filter(image, ksize):
    """
    Removes noise by replacing each pixel with the median of its surrounding pixels. Suitable for salt and pepper noise.
    Read more: https://en.wikipedia.org/wiki/Median_filter
    """
    denoised_image = cv2.medianBlur(image.astype(np.uint8), ksize)
    return denoised_image


def denoise_with_gaussian_blur(image, ksize):
    """
    Smoothens image using a Gaussian filter, effectively reducing noise but also blurring edges.
    Read more: https://en.wikipedia.org/wiki/Gaussian_blur
    """
    denoised_image = cv2.GaussianBlur(image, (ksize, ksize), 0)
    return denoised_image


def denoise_with_wiener(image, ksize):
    """
    Applies Wiener filter to reduce noise by considering local image variance. Good for various noise types, preserves edges better.
    Read more: https://en.wikipedia.org/wiki/Wiener_filter
    """
    return wiener(image.astype(np.float32), (ksize, ksize)).astype(np.uint8)

In [ ]:
def simulate_denoising(target_psnrs, denoise_ksize):
    results = []

    for target_psnr in target_psnrs:
        img_noisy = add_awgn_noise_with_target_psnr(lenna_gray, target_psnr)
        
        img_denoised_wiener = denoise_with_wiener(img_noisy, denoise_ksize)
        img_denoised_gaussian = denoise_with_gaussian_blur(img_noisy, denoise_ksize)
        img_denoised_median = denoise_with_median_filter(img_noisy, denoise_ksize)
        cv2.imwrite('test.png', img_noisy)
        cv2.imwrite('test2.png', img_denoised_wiener)

        psnr_noisy = psnr_ski(lenna_gray, img_noisy)
        psnr_wiener = psnr_ski(lenna_gray, img_denoised_wiener)
        psnr_gaussian = psnr_ski(lenna_gray, img_denoised_gaussian)
        psnr_median = psnr_ski(lenna_gray, img_denoised_median)

        results.append(
            {
                "target_psnr": target_psnr,
                "psnr_noisy": psnr_noisy,
                "psnr_wiener": psnr_wiener,
                "psnr_gaussian": psnr_gaussian,
                "psnr_median": psnr_median,
            }
        )

    df = pd.DataFrame(results)

    fig, ax = plt.subplots()
    
    sns.lineplot(data=df, x="target_psnr", y="psnr_noisy", label="actual psnr", ax=ax)
    sns.lineplot(data=df, x="target_psnr", y="psnr_wiener", label="wiener", ax=ax)
    sns.lineplot(data=df, x="target_psnr", y="psnr_gaussian", label="gaussian", ax=ax)
    sns.lineplot(data=df, x="target_psnr", y="psnr_median", label="median", ax=ax)
    ax.set_xlim(min(target_psnrs), max(target_psnrs))
    ax.legend()
    plt.show()


Here you can play around by putting different ranges of PSNR and denoising kernel size to see the difference between denoising methods and its performance.

Note: in this simulation we are applying only AWGN.

In [ ]:
simulate_denoising(range(8, 50), 5)

---
## Part B, hasihing techniques and theirs robustness (5 points)

Split the image into ```h×w``` blocks by dividing its dimensions. Hint: use ```view_as_windows``` from ```skimage.util```.

Compute the overall image mean. 

Compare each pixel's value against this mean: assign `1` if the pixel's value is greater than the mean, else `0`.

In [ ]:
def mean_intensity_hash(image, h=8, w=8):
    """
    Computes a hash of an image by splitting it into h x w blocks and comparing
    the mean intensity of each block to the overall image mean.

    Parameters:
    - image: the grayscale image.
    - h: Integer, number of blocks along the height.
    - w: Integer, number of blocks along the width.

    Returns:
        1D array of length h * w, containing 0s and 1s representing the hash.

    Read more about image hashing: https://en.wikipedia.org/wiki/Perceptual_hashing
    """

    # YOUR CODE HERE
    raise NotImplementedError()
    # Flatten the 2D hash values array to 1D
    return hash_values.flatten()

Compute the 2D FFT of the grayscale image and shift the zero-frequency component to the center. Hint: use `np.fft.fft2` and `np.fft.fftshift`.

Obtain the magnitude spectrum and resize it to $(hash\_size \times hash\_size)$.

Compare each element to the overall mean of this resized spectrum: assign 1 if greater, else 0.

In [ ]:
def fft_image_hash(image, hash_size=8):
    """
    Computes an image hash based on the FFT of the image.

    Parameters:
    - image: the grayscale image.
    - hash_size: Integer, the width and height of the hash in bits.

    Returns:
        1D array of length hash_size ^ 2, containing 0s and 1s representing the hash.

    Read more about FFT: https://en.wikipedia.org/wiki/Fast_Fourier_transform
    Read more about image hashing: https://en.wikipedia.org/wiki/Perceptual_hashing
    """
    if len(image.shape) > 2:
        raise ValueError("Image must be grayscale")

    # YOUR CODE HERE
    raise NotImplementedError()

    return hash_array

Convert the image to a bytes representation with. Hint: use `tobytes()`.

Use Python's `hashlib.md5()` to compute the MD5 hash of these bytes. 

Convert the hash to a hexadecimal string with. Hint: use `.hexdigest()`.

In [ ]:
def md5_image_hash(image):
    """
    Computes an MD5 hash for an image.

    Parameters:
    - image: any image.

    Returns:
        A hexadecimal string representing the MD5 hash of the image.

    Read more about MD5: https://en.wikipedia.org/wiki/MD5
    """
    # YOUR CODE HERE
    raise NotImplementedError()

    return hash_hex

In [ ]:
hash_mean = mean_intensity_hash(lenna_gray)
hash_fft = fft_image_hash(lenna_gray)
hash_md5 = md5_image_hash(lenna_gray)
print(hash_mean, "\n", hash_fft, "\n", hash_md5)

In [ ]:
hash_md5 = md5_image_hash(lenna_gray)
lenna_gray_mod = lenna_gray.copy()
lenna_gray_mod[0, 0] = 255
hash_md5_singlepixel = md5_image_hash(lenna_gray_mod)
print(hash_md5, '\n', hash_md5_singlepixel)

In [ ]:

results = []
for target_psnr in range(1, 50):
    lenna_gray_n = add_awgn_noise_with_target_psnr(lenna_gray, target_psnr)
    results.append(
        {
            "target_psnr": target_psnr,
            "p_ber_mih": (mean_intensity_hash(lenna_gray_n) != hash_mean).mean(),
            "p_ber_fft": (fft_image_hash(lenna_gray_n) != hash_fft).mean(),
        }
    )

df = pd.DataFrame(results)
plt.figure(figsize=(10, 5))
sns.lineplot(df, x='target_psnr', y='p_ber_mih', label='Mean intensity hash')
# sns.lineplot(df, x='target_psnr', y='p_ber_fft', label='FFT hash')
plt.legend()

---
## Part C, steganography (5 points)

To implement the following stegonography algorithm in the most efficient manner it is crucial to understand bitwise operations (`&`, `>>`, `+`) and how they manipulate binary data. Remember, working with images as arrays allows you to apply operations to whole channels at once.

1. **Masking Cover Image**: Create a mask to zero out the least significant bits (LSBs) in the cover image for the specified number of bits per channel. Hint: use `<<` operator.
   
2. **Extracting Secret Image Bits**: Shift the secret image bits right so that the bits you want to hide become the LSBs, and then mask these bits to isolate them. Hint: use `>> &` operators.

3. **Combining Images**: Add the modified secret image bits to the masked cover image, effectively embedding the secret image within the cover image's LSBs.



In [ ]:
def hide_image_lsb(cover_image, secret_image, bits_per_channel):
    """
    Hides a secret image inside a cover image using the least significant bit method.

    Parameters:
    - cover_image: RGB uint8 cover image.
    - secret_image: RGB uint8 secret image to hide.
    - bits_per_channel: List of integers, the number of least significant bits to use per channel.

    Returns:
        Cover image with the secret image hidden inside.

    Read more about steganography: https://en.wikipedia.org/wiki/Steganography
    Read more about the least significant bit method: https://en.wikipedia.org/wiki/Least_significant_bit
    """
    if not (0 < len(bits_per_channel) == 3):
        raise ValueError("bits_per_channel must be a list of 3 positive integers")

    if cover_image.shape != secret_image.shape:
        raise ValueError(
            "Both images must have the same dimensions and number of channels"
        )

    # YOUR CODE HERE
    raise NotImplementedError()
    return combined_image

If you successfully implemented the algorithm, you should be able to extract the secret image from the stego image and recover the original secret image.

Implement a function which takes the cover image, secret image, and the number of bits per channel to use for the steganography.

The function should return the stego image and the extracted secret image.

In [ ]:
def extract_images_lsb(combined_image, bits_per_channel):
    """
    Extracts the cover and secret images from a combined image.

    Parameters:
    - combined_image: the combined image from which to extract the cover and secret images.
    - bits_per_channel: List of integers, the number of least significant bits used per channel.

    Returns:
        A tuple of two NumPy images: the extracted cover image and the extracted secret image.
    """
    if not (0 < len(bits_per_channel) == 3):
        raise ValueError("bits_per_channel must be a list of 3 positive integers")

    # YOUR CODE HERE
    raise NotImplementedError()

    return cover_image, secret_image

In contrast to previous implementation here you specify just the number of bits per **all** channels.

In [ ]:
def extract_lsb_blind(image, num_lsb):
    """
    Extracts the cover and secret images from a combined image.

    Parameters:
    - image: the combined image from which to extract the secret images.
    - num_lsb: Number of least significant bits used for all channels.

    Returns:
        The extracted secret image.
    """
    # YOUR CODE HERE
    raise NotImplementedError()

    return secret_image

In [ ]:
lsbs = [1, 2, 3]
stego = hide_image_lsb(lenna, baboon, lsbs)
lenna_recovered, baboon_recovered = extract_images_lsb(stego, lsbs)
lenna_recovered, baboon_recoevered_blind = extract_images_lsb(stego, [3, 3, 3])
# baboon_recoevered_blind = extract_lsb_blind(stego, 3)
plt.figure(figsize=(20, 4))
plt.imshow(
    np.hstack(
        [
            # stego,
            # lenna,
            # lenna_recovered,
            # baboon,
            baboon_recovered,
            baboon_recoevered_blind,
        ]
    )
)

In [ ]:
def dct2(a):
    """Perform a 2D DCT."""
    return dct(dct(a.T, norm="ortho").T, norm="ortho")


def idct2(a):
    """Perform a 2D Inverse DCT."""
    return idct(idct(a.T, norm="ortho").T, norm="ortho")

Apply Discrete Cosine Transform (DCT) (`dct2` function) to both the cover and secret images.

Mix the DCT coefficients of the secret image into those of the cover image using the blending factor alpha.

The secret is hidden by altering the frequency components.

Then, apply the inverse DCT (`idct2`) to transform back to the pixel domain, obtaining the stego image.

Finally, normalize the pixel values to ensure they remain in the valid image range.

In [ ]:
def hide_image_dct(image_cover, image_secret, alpha=0.05):
    """
    Hide a secret image within a cover image using DCT.

    Parameters:
    - cover_image: Cover image.
    - secret_image: Secret image.
    - alpha: The blending factor indicating how much of the secret image to mix in.

    Returns:
        Stego image.

    Read more about DCT: https://en.wikipedia.org/wiki/Discrete_cosine_transform

    """
    # YOUR CODE HERE
    raise NotImplementedError()
    return stego_img

Apply Discrete Cosine Transform (DCT) to both the stego image and the original cover image.

Subtract the cover's DCT coefficients from the stego's and divide by alpha to isolate the DCT coefficients of the secret image.

Perform the inverse DCT on these extracted coefficients to convert back to the pixel domain, retrieving the secret image.

Finally, normalize the pixel values to ensure they're within the valid image range.

In [ ]:
def extract_secret_dct(image_stego, image_cover, alpha=0.05):
    """
    Extract the secret image from a stego image.

    Parameters:
    - image_stego: Stego image.
    - image_cover: Cover image.
    - alpha: The blending factor used for hiding the secret image.

    Returns:
        A numpy array of the extracted secret image.
    """
    # YOUR CODE HERE
    raise NotImplementedError()
    return secret_img_extracted

In [ ]:
lenna_gray
baboon_gray
a = 0.05
stego_dct = hide_image_dct(lenna_gray, baboon_gray, a)
baboon_recovered = extract_secret_dct(stego_dct, lenna_gray, a)
plt.figure(figsize=(20, 4))
plt.imshow(
    np.hstack(
        [
            lenna_gray,
            baboon_gray,
            stego_dct,
            minmax_norm(stego_dct - lenna_gray) * 255,
            baboon_recovered,
        ]
    ),
    cmap="gray",
)
calculate_psnr(
    minmax_norm(baboon_gray) * 255, minmax_norm(stego_dct - lenna_gray) * 255
)

In [ ]:
minmax_norm(stego_dct - lenna_gray) * 255